In [1]:
from google.colab import files

# This will open a file upload button in your browser
uploaded = files.upload()

Saving appointments_clean.csv to appointments_clean.csv


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments_clean.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())

Dataset shape: (990, 16)
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show        

In [3]:
# TASK 3.1: Select features, split data and calculate baseline
# keep appontment_id separate
appointment_ids = df["appointment_id"].copy()

# create the feature columns
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# convert reminder_sent into number
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# apply one-hot encoding
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

print("Feature columns:")
print(X.columns.tolist())

print("Number of features:", X.shape[1])

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Number of features: 11


In [4]:
# create the target variable
y = df["no_show"]

print(y.value_counts())

no_show
0    772
1    218
Name: count, dtype: int64


In [5]:
# split into training and testing data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("No-shows in test set:", int(y_test.sum()))

Training rows: 792
Testing rows: 198
No-shows in test set: 49


In [6]:
# calculate baseline accuracy
baseline_accuracy = (y_test == 0).mean()

print("Baseline accuracy:", baseline_accuracy)

Baseline accuracy: 0.7525252525252525


In [7]:
# TASK 3.2: Train Logistic Regression
# IMPORT REQUIRED TOOLS
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# scale the feature
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train the Logistic Regression model
model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

# predict the test data
y_pred = model.predict(X_test_s)

y_prob = model.predict_proba(X_test_s)[:, 1]


# calculate evalution matrix
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


Accuracy: 0.7878787878787878
Confusion Matrix:
[[139  10]
 [ 32  17]]
Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1 Score: 0.4473684210526316
ROC-AUC: 0.788933022873579


In [8]:
# display the model coefficient
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print(coefficients.sort_values(
    by="Coefficient",
    ascending=False
))

                    Feature  Coefficient
4            prior_no_shows     0.649612
3                 lead_days     0.614471
2                   fee_inr     0.452761
1               distance_km     0.253448
7     booking_channel_Phone     0.215881
9   visit_type_Consultation     0.015658
10     visit_type_Follow-up    -0.015658
6       booking_channel_App    -0.046804
8   booking_channel_Walk-In    -0.186364
0                       age    -0.253880
5             reminder_sent    -0.580669


In [9]:
# calculate the reminder odds ratio
reminder_index = list(X.columns).index("reminder_sent")

reminder_coef = model.coef_[0][reminder_index]
reminder_scale = scaler.scale_[reminder_index]

scaled_odds_ratio = np.exp(reminder_coef)
actual_odds_ratio = np.exp(reminder_coef / reminder_scale)

print("Reminder coefficient:", reminder_coef)
print("Odds ratio per 1 standard deviation:", scaled_odds_ratio)
print("Odds ratio for No to Yes reminder:", actual_odds_ratio)

Reminder coefficient: -0.5806686033659922
Odds ratio per 1 standard deviation: 0.559524141750829
Odds ratio for No to Yes reminder: 0.3093980903124433


In [10]:
# TASK 3.3: Decision Tree and overfiting
# import decision tree
from sklearn.tree import DecisionTreeClassifier

# train full depth tree
tree_full = DecisionTreeClassifier(random_state=42)

tree_full.fit(X_train, y_train)

train_pred_full = tree_full.predict(X_train)
test_pred_full = tree_full.predict(X_test)

print("Full Tree Training Accuracy:",
      accuracy_score(y_train, train_pred_full))

print("Full Tree Testing Accuracy:",
      accuracy_score(y_test, test_pred_full))

print("Tree Depth:", tree_full.get_depth())

Full Tree Training Accuracy: 1.0
Full Tree Testing Accuracy: 0.6717171717171717
Tree Depth: 15


In [11]:
# train depth-3 tree
tree_3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_3.fit(X_train, y_train)

train_pred_3 = tree_3.predict(X_train)
test_pred_3 = tree_3.predict(X_test)

print("Depth-3 Training Accuracy:",
      accuracy_score(y_train, train_pred_3))

print("Depth-3 Testing Accuracy:",
      accuracy_score(y_test, test_pred_3))

Depth-3 Training Accuracy: 0.821969696969697
Depth-3 Testing Accuracy: 0.7777777777777778


In [12]:
#TASK 3.4: Random Forest and model comparison
# Import and train Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [13]:
# Make predictions
rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:, 1]

In [14]:
# calculate matrix
print("Random Forest Training Accuracy:",
      accuracy_score(y_train, rf.predict(X_train)))

print("Random Forest Testing Accuracy:",
      accuracy_score(y_test, rf_pred))

print("Random Forest Recall:",
      recall_score(y_test, rf_pred, zero_division=0))

print("Random Forest ROC-AUC:",
      roc_auc_score(y_test, rf_prob))

Random Forest Training Accuracy: 1.0
Random Forest Testing Accuracy: 0.7474747474747475
Random Forest Recall: 0.30612244897959184
Random Forest ROC-AUC: 0.7057252431173813


In [15]:
# find the top three important feature
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(3))

       Feature  Importance
3    lead_days    0.219905
0          age    0.196788
1  distance_km    0.194739


In [16]:
# Create a comparison table
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, test_pred_3),
        accuracy_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, test_pred_3, zero_division=0),
        recall_score(y_test, rf_pred, zero_division=0)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, tree_3.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, rf_prob)
    ]
})

print(comparison.round(3))

                     Model  Accuracy  Recall  ROC-AUC
0      Logistic Regression     0.788   0.347    0.789
1  Decision Tree (Depth 3)     0.778   0.224    0.700
2            Random Forest     0.747   0.306    0.706


In [17]:
# TASK 3.5: Tune, select the final model and choose a threshold
# Tune Random Forest using GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7673242251739405


In [18]:
# Compare models using training-only cross-validation
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Logistic Regression
lr_cv = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Decision Tree
tree_cv = cross_val_score(
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Random Forest
rf_cv = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

# Tuned Random Forest
tuned_rf_cv = grid_search.best_score_

print("Logistic Regression CV AUC:", lr_cv)
print("Depth-3 Tree CV AUC:", tree_cv)
print("Random Forest CV AUC:", rf_cv)
print("Tuned Random Forest CV AUC:", tuned_rf_cv)

Logistic Regression CV AUC: 0.7737139899948249
Depth-3 Tree CV AUC: 0.6784494968661952
Random Forest CV AUC: 0.7466024782933701
Tuned Random Forest CV AUC: 0.7673242251739405


In [19]:
# Choose the prediction threshold using training data
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

In [20]:
# compare two thresholds:
thresholds = [0.50, 0.35]

for threshold in thresholds:

    predictions = (oof >= threshold).astype(int)

    print("\nThreshold:", threshold)

    print("Precision:",
          precision_score(y_train, predictions, zero_division=0))

    print("Recall:",
          recall_score(y_train, predictions, zero_division=0))

    print("F1 Score:",
          f1_score(y_train, predictions, zero_division=0))

    print("Appointments Flagged:",
          predictions.sum())


Threshold: 0.5
Precision: 0.5753424657534246
Recall: 0.2485207100591716
F1 Score: 0.34710743801652894
Appointments Flagged: 73

Threshold: 0.35
Precision: 0.512987012987013
Recall: 0.46745562130177515
F1 Score: 0.4891640866873065
Appointments Flagged: 154


In [21]:
# Train the final model and evaluate it once on the test set
lr_pipe.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

In [22]:
# predicted probabilities for the test set:
probs = lr_pipe.predict_proba(X_test)[:, 1]

In [23]:
# selected threshold:
final_threshold = 0.35

final_predictions = (probs >= final_threshold).astype(int)

In [24]:
# Calculate the final metrics:
print("Final Model: Logistic Regression")
print("Threshold:", final_threshold)

print("Precision:",
      precision_score(y_test, final_predictions, zero_division=0))

print("Recall:",
      recall_score(y_test, final_predictions, zero_division=0))

print("F1 Score:",
      f1_score(y_test, final_predictions, zero_division=0))

print("ROC-AUC:",
      roc_auc_score(y_test, probs))

print("Number of Appointments Flagged:",
      final_predictions.sum())

Final Model: Logistic Regression
Threshold: 0.35
Precision: 0.5714285714285714
Recall: 0.5714285714285714
F1 Score: 0.5714285714285714
ROC-AUC: 0.788933022873579
Number of Appointments Flagged: 49


In [25]:
# Save test risk scores to CSV
test_risk_scores = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": probs,
    "flag": final_predictions
})

test_risk_scores.to_csv(
    "test_risk_scores.csv",
    index=False
)

print(test_risk_scores.head())

print("Saved successfully!")
print("Total test rows:", len(test_risk_scores))
print("Total flagged:", test_risk_scores["flag"].sum())

  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0
Saved successfully!
Total test rows: 198
Total flagged: 49


In [26]:
# Verify your saved CSV
check = pd.read_csv("test_risk_scores.csv")

print("Shape:", check.shape)
print("Columns:", check.columns.tolist())
print("Missing values:")
print(check.isnull().sum())

print("Flag counts:")
print(check["flag"].value_counts())

Shape: (198, 4)
Columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']
Missing values:
appointment_id    0
no_show           0
risk_prob         0
flag              0
dtype: int64
Flag counts:
flag
0    149
1     49
Name: count, dtype: int64


### Machine Learning Conclusion

In this task, I developed and evaluated Logistic Regression, Decision Tree, and Random Forest models to predict clinic appointment no-shows.

Logistic Regression performed best in the training cross-validation comparison with a 5-fold cross-validated ROC-AUC of **0.774**, compared to **0.678** for the Decision Tree and **0.767** for the tuned Random Forest. Therefore, I selected Logistic Regression as the final model.

I compared two prediction thresholds using training-only out-of-fold predictions: 0.50 and 0.35. I selected a probability threshold of **0.35** because it improved the recall to **46.7%** (compared to 24.9% at a threshold of 0.50) and yielded a better F1 score of **0.489** (compared to 0.347 at a threshold of 0.50), allowing us to catch more potential no-shows.

Evaluating the final model on the unseen test dataset (198 appointments) with the selected **0.35** threshold produced the following test results:
- **ROC-AUC**: 0.789
- **Accuracy**: 78.8%
- **Precision**: 57.1%
- **Recall**: 57.1%
- **F1 Score**: 0.571
- **Appointments Flagged for Follow-up**: 49 out of 198 (with 149 classified as likely to attend)

The final risk scores have been successfully validated and saved to `test_risk_scores.csv` with no missing values. In a real clinic, this threshold can be dynamically adjusted based on the staff's capacity to handle follow-up calls and the relative cost of an empty appointment slot versus the cost of a reminder intervention.